<img src="https://github.com/jelenavasic009-png/DSB-Unit-02/blob/main/assets/ga-logo.png?raw=1" style="float: left; margin: 20px; height: 55px">

# Exercise: descriptive statistics in `pandas`

_Authors: David Asboth_

---

## Learning Objectives

After this lesson, students will be able to:

- Explore descriptive statistics of a data set using pandas

# Lesson Guide

This notebook helps students practice:

- Calculating descriptive statistics in `pandas`
- Doing group-level analysis in `pandas`
    - Doing "split-apply-combine" with `groupby`
    - Aggregating and summarizing data

# Scenario

You're an analyst working for CoolBricks - a new toy company who wants to compete with LEGO by producing their own bricks and sets. To better understand what kind of sets to create and at what price points, you are tasked with analyzing LEGO's catalogue over time.

The data comes from Kaggle: [https://www.kaggle.com/datasets/alexracape/lego-sets-and-prices-over-time](https://www.kaggle.com/datasets/alexracape/lego-sets-and-prices-over-time)

We are going to use `pandas` to read and explore this data.

The question we want to answer is: **How do LEGO products vary across different types/themes?**

## Guided questions

***Remember to check back to notebooks 1 and 2 - they contain everything you need!***

Let's import the data and preview it.

*Note: if you place the `lego.csv` file in the same folder as this notebook, you'll need to change the filepath to just `"lego.csv"`*

In [ ]:
import pandas as pd

lego = pd.read_csv("../data/lego.csv")
lego.head()

,Set_ID,Name,Year,Theme,Theme_Group,Subtheme,Category,Packaging,Num_Instructions,Availability,Pieces,Minifigures,Owned,Rating,USD_MSRP,Total_Quantity,Current_Price
0,75-1,PreSchool Set,1975,PreSchool,Pre-school,NaN,Normal,{Not specified},0,{Not specified},16.0,NaN,10.0,0.0,NaN,NaN,NaN
1,77-1,PreSchool Set,1975,PreSchool,Pre-school,NaN,Normal,{Not specified},0,{Not specified},20.0,NaN,11.0,0.0,NaN,NaN,NaN
2,077-1,Pre-School Set,1975,Duplo,Pre-school,NaN,Normal,{Not specified},0,{Not specified},21.0,NaN,10.0,0.0,NaN,0.0,NaN
3,78-1,PreSchool Set,1975,PreSchool,Pre-school,NaN,Normal,{Not specified},0,{Not specified},32.0,NaN,8.0,0.0,NaN,NaN,NaN
4,78-3,Basic Set,1975,Samsonite,Vintage,Basic set,Normal,Box,0,{Not specified},330.0,NaN,10.0,0.0,NaN,0.0,NaN


#### 1. Find the "theme group" that has the biggest sets, i.e. that has the highest average number of pieces

In [21]:
# Group by theme group, calculate the average number of pieces, and sort in descending order
avg_pieces_by_theme = lego.groupby("Theme_Group")["Pieces"].mean().reset_index()
top_theme_group = avg_pieces_by_theme.sort_values(by="Pieces", ascending=False).iloc[0]

print(f"The theme group with the biggest sets is: {top_theme_group['Theme_Group']}")
print(f"Highest average pieces: {top_theme_group['Pieces']:.2f}")


The theme group with the biggest sets is: Art and crafts
Highest average pieces: 775.28


#### 2. How does that group compare to the average number of pieces in the entire data set?

In [ ]:
# 1. Calculate the overall average pieces across all sets
overall_avg = lego['Pieces'].mean()

# 2. Get the top theme group's average (from the previous step)
top_theme_avg = top_theme_group['Pieces']

# 3. Calculate the difference
difference = top_theme_avg - overall_avg
percentage_larger = (difference / overall_avg) * 100

print(f"Top Theme Group Average: {top_theme_avg:.2f} Pieces")
print(f"Overall Dataset Average: {overall_avg:.2f} Pieces")
print(f"The top theme group is, on average, {difference:.2f} pieces larger ({percentage_larger:.1f}% higher) than the dataset average.")


Top Theme Group Average: 775.28 Pieces
Overall Dataset Average: 237.94 Pieces
The top theme group is, on average, 537.34 pieces larger (225.8% higher) than the dataset average.


#### 3. Within this theme group, how does the number of pieces compare per individual theme?

*Hint: you'll need to filter your data before doing the aggregation!*

In [ ]:
# 1. Identify the top theme group name from your previous analysis
# (For example purposes, let's assume it was saved in top_theme_group['themeGroup'])
target_group = top_theme_group['Theme_Group']

# 2. Filter the dataset for ONLY this theme group
filtered_lego = lego[lego['Theme_Group'] == target_group]

# 3. Group by the individual 'theme' column and calculate the average pieces
theme_comparison = filtered_lego.groupby('Theme')['Pieces'].mean().reset_index()

# 4. Sort from biggest average to smallest
theme_comparison = theme_comparison.sort_values(by='Pieces', ascending=False)

# Display the breakdown
print(f"--- Breakdown of Themes within '{target_group}' ---")
print(theme_comparison.to_string(index=False))


--- Breakdown of Themes within 'Art and crafts' ---
Theme      Pieces
  Art 3836.692308
 Dots  283.938272


#### 4. Find the most expensive set in the "Creator Expert" theme

In [22]:
# 1. Filter the dataset for the "Creator Expert" theme
creator_expert_sets = lego[lego['Theme'] == 'Creator Expert']

# 2. Find the row with the maximum price
# (Using .idxmax() locates the exact index of the highest value)
most_expensive_set = creator_expert_sets.loc[creator_expert_sets['Current_Price'].idxmax()]

# Display the results
print("--- Most Expensive Creator Expert Set ---")
print(f"Set Name: {most_expensive_set['Name']}")
print(f"Set Number: {most_expensive_set['Set_ID']}")
print(f"Price: ${most_expensive_set['Current_Price']:.2f}")
print(f"Piece Count: {most_expensive_set['Pieces']}")


--- Most Expensive Creator Expert Set ---
Set Name: Brick Bank
Set Number: 10251-1
Price: $597.62
Piece Count: 2380.0


#### 5. BONUS - can you figure out the average price AND the maximum number of pieces for each **theme group** in a single command?

*Hint: look at the `.agg` command...*

In [23]:
# Group by theme group and apply distinct aggregations to price and pieces
theme_group_summary = lego.groupby('Theme_Group').agg({'Current_Price': 'mean', 'Pieces': 'max'})

# Optional: Rename columns to make the output clean and easy to read
theme_group_summary = theme_group_summary.rename(columns={'Current_Price': 'avg_price', 'Pieces': 'max_pieces'})

print(theme_group_summary)


                   avg_price  max_pieces
Theme_Group                             
Action/Adventure   87.133444      6163.0
Art and crafts     30.882232     11695.0
Basic              22.600684      2847.0
Constraction       80.970243       856.0
Educational       133.753232      4900.0
Girls              57.785984       450.0
Historical        210.434185      1601.0
Junior             32.721408       480.0
Licensed          112.382881      7541.0
Miscellaneous      44.914558      4502.0
Model making      122.405318     10001.0
Modern day         64.486840      2010.0
Pre-school         36.467305       640.0
Racing             55.071398      1484.0
Technical         113.706339      4108.0
Vintage           154.824400      1112.0


## Reflection

Based on your initial findings, what answers can you give to the initial business question?

> **How do LEGO products vary across different types/themes?**